# LeetCode 84 — Largest Rectangle in Histogram (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:

```

Đây là một bài **Hard** rất nổi tiếng, và cũng là bài "kinh điển" để học **monotonic stack** (ngăn xếp đơn điệu). Đề chỉ vài dòng, brute force thì ai cũng nghĩ ra, nhưng để chạy kịp với `10^5` cột ta cần một ý tưởng thật sự đẹp. Notebook này đi **từng bậc thang**, mỗi bậc chỉ sửa đúng một chỗ "vướng" của bậc trước.

---

## Sau notebook này, bạn sẽ nắm được

1. Hình dung bài toán bằng **hình vẽ ASCII** histogram và tự đếm diện tích bằng tay.
2. **Cách 1 — Brute force mọi đoạn `(i, j)`**: bản ngây thơ `O(n³)` và bản "min chạy" `O(n²)`.
3. **Cách 2 — Lan trái/phải từ mỗi cột**: đổi góc nhìn "mỗi hình chữ nhật tốt nhất có một **cột thấp nhất**", vẫn `O(n²)` ở worst case.
4. **Cách 3 — Chia để trị** theo cột thấp nhất (đệ quy): nhanh khi cắt cân, nhưng worst case `O(n²)` và có thể tràn đệ quy.
5. **Cách 4 — Mảng `left_smaller` / `right_smaller`** bằng monotonic stack hai lượt: `O(n)`.
6. **Cách 5 — Monotonic stack một lượt với sentinel `0`** (cách chính): `O(n)` thời gian, trace từng bước, hiểu thật kỹ vì sao `width = i - stack[-1] - 1`.
7. Code nộp LeetCode, kiểm thử đối chiếu chéo, lỗi hay gặp, mẹo nhận dạng pattern, 6 bài tập (có LeetCode 85 và 1793) và đáp án ở cuối.

## 1. Đề bài (theo LeetCode)

Cho mảng số nguyên `heights` biểu diễn chiều cao các cột của một **histogram** (biểu đồ cột), trong đó **mỗi cột có chiều rộng 1**. Hãy trả về **diện tích của hình chữ nhật lớn nhất** nằm trong histogram.

**Example 1**

```text
Input:  heights = [2,1,5,6,2,3]
Output: 10
Explanation: The above is a histogram where width of each bar is 1.
The largest rectangle is shown in the red area, which has an area = 10 units.
```

**Example 2**

```text
Input:  heights = [2,4]
Output: 4
```

**Constraints**

- `1 <= heights.length <= 10^5`
- `0 <= heights[i] <= 10^4`

Vài điều cần để ý ngay từ đầu:

- Hàm trả về **một số nguyên** (diện tích), không phải vị trí hình chữ nhật.
- Hình chữ nhật phải **nằm sát đáy** (trục hoành) và **nằm gọn bên trong** histogram: nó phủ một dãy cột **liền nhau**, và không được "thò" lên cao hơn bất kỳ cột nào nó phủ.
- Chiều cao có thể bằng `0` — một cột cao 0 giống như một "vết cắt" chia histogram thành hai phần.
- `n` lên tới `10^5` → lời giải `O(n²)` (khoảng `10^10` phép tính) chắc chắn **TLE** (Time Limit Exceeded — quá thời gian). Ta cần khoảng `O(n)` hoặc `O(n log n)`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

heights1 = [2, 1, 5, 6, 2, 3]
heights2 = [2, 4]

print("Example 1:", heights1, "-> mong đợi 10")
print("Example 2:", heights2, "-> mong đợi 4")
print("Số cột của example 1:", len(heights1))

Example 1: [2, 1, 5, 6, 2, 3] -> mong đợi 10
Example 2: [2, 4] -> mong đợi 4
Số cột của example 1: 6


## 2. Vẽ histogram ra cho dễ thấy

Với bài hình học như thế này, **vẽ hình** giúp ích hơn mọi công thức. Quy ước:

- mỗi cột rộng 2 ký tự cho dễ nhìn, `##` là một ô của cột,
- `@@` là ô thuộc **hình chữ nhật** ta đang xét,
- hàng dưới cùng là **index** (chỉ số) của cột.

Example 1 `[2,1,5,6,2,3]`:

```text
 6 |      ##    
 5 |    ####    
 4 |    ####    
 3 |    ####  ##
 2 |##  ########
 1 |############
   +------------
    0 1 2 3 4 5 
```

Hình chữ nhật lớn nhất (diện tích 10) phủ cột 2 và 3, cao 5:

```text
 6 |      ##    
 5 |    @@@@    
 4 |    @@@@    
 3 |    @@@@  ##
 2 |##  @@@@####
 1 |####@@@@####
   +------------
    0 1 2 3 4 5 
```

Vì sao không cao 6? Vì cột 2 chỉ cao 5 — hình chữ nhật cao 6 sẽ "thò" ra ngoài cột 2. Vì sao không rộng 3 (thêm cột 4)? Vì cột 4 chỉ cao 2, khi đó hình chữ nhật chỉ còn cao 2, diện tích `2 × 3 = 6 < 10`.

Cell dưới là hàm vẽ. Bạn **không cần** hiểu hết chi tiết; chỉ cần biết cách gọi: `draw_histogram(heights)` hoặc `draw_histogram(heights, rect=(left, right, h))`.

In [2]:
def draw_histogram(heights, rect=None):
    """Vẽ histogram nằm dọc.
    '##' = ô thuộc cột, '@@' = ô thuộc hình chữ nhật rect = (left, right, h)
    (rect phủ các cột left..right, cao từ đáy lên tới h)."""
    top = max(heights) if heights else 0
    for level in range(top, 0, -1):            # vẽ từ hàng trên cùng xuống
        row = ""
        for c, h in enumerate(heights):
            if rect is not None and rect[0] <= c <= rect[1] and level <= rect[2]:
                row += "@@"
            elif h >= level:
                row += "##"
            else:
                row += "  "
        print(f"{level:>2} |{row}")
    print("   +" + "--" * len(heights))
    print("    " + "".join(f"{c % 10} " for c in range(len(heights))))


print("Example 1:")
draw_histogram(heights1)
print("\nHình chữ nhật lớn nhất: cột 2..3, cao 5")
draw_histogram(heights1, rect=(2, 3, 5))
print("\nExample 2:")
draw_histogram(heights2, rect=(0, 1, 2))   # rộng 2, cao 2 -> diện tích 4

Example 1:
 6 |      ##    
 5 |    ####    
 4 |    ####    
 3 |    ####  ##
 2 |##  ########
 1 |############
   +------------
    0 1 2 3 4 5 

Hình chữ nhật lớn nhất: cột 2..3, cao 5
 6 |      ##    
 5 |    @@@@    
 4 |    @@@@    
 3 |    @@@@  ##
 2 |##  @@@@####
 1 |####@@@@####
   +------------
    0 1 2 3 4 5 

Example 2:
 4 |  ##
 3 |  ##
 2 |@@@@
 1 |@@@@
   +----
    0 1 


## 3. Quan sát đầu tiên: một dãy cột liền nhau "cho phép" cao bao nhiêu?

Chọn một **đoạn cột liền nhau** từ cột `i` tới cột `j` (gồm cả hai đầu). Hình chữ nhật phủ đoạn này:

- **rộng** `j - i + 1` (số cột trong đoạn),
- **cao tối đa** bằng **cột thấp nhất** trong đoạn, tức `min(heights[i..j])` — cao hơn thì sẽ thò ra khỏi cột thấp đó.

```text
area(i, j) = min(heights[i..j]) × (j - i + 1)
```

**Ví dụ số** với `heights = [2,1,5,6,2,3]`:

| đoạn `i..j` | các cột | min | rộng | diện tích |
|---|---|---:|---:|---:|
| `2..3` | `[5, 6]` | 5 | 2 | **10** |
| `2..4` | `[5, 6, 2]` | 2 | 3 | 6 |
| `4..5` | `[2, 3]` | 2 | 2 | 4 |
| `2..5` | `[5, 6, 2, 3]` | 2 | 4 | 8 |
| `0..5` | cả mảng | 1 | 6 | 6 |
| `3..3` | `[6]` | 6 | 1 | 6 |

Bài toán trở thành: **tìm đoạn `i..j` làm `min × độ dài` lớn nhất.** Mọi cách giải phía sau chỉ là các cách khác nhau để tìm đoạn đó nhanh hơn.

In [3]:
def area_of_range(heights, i, j):
    # heights[i:j+1] lấy các phần tử index i, i+1, ..., j (slicing KHÔNG lấy phần tử j+1)
    return min(heights[i:j + 1]) * (j - i + 1)

for i, j in [(2, 3), (2, 4), (4, 5), (2, 5), (0, 5), (3, 3)]:
    print(f"đoạn {i}..{j}: cột {heights1[i:j + 1]}, diện tích = {area_of_range(heights1, i, j)}")

đoạn 2..3: cột [5, 6], diện tích = 10
đoạn 2..4: cột [5, 6, 2], diện tích = 6
đoạn 4..5: cột [2, 3], diện tích = 4
đoạn 2..5: cột [5, 6, 2, 3], diện tích = 8
đoạn 0..5: cột [2, 1, 5, 6, 2, 3], diện tích = 6
đoạn 3..3: cột [6], diện tích = 6


## 4. Cách 1 — Brute force: thử mọi đoạn `(i, j)`

### Kiến thức nền: hai vòng `for` lồng nhau để liệt kê mọi đoạn

Muốn thử **mọi** đoạn liền nhau, ta chọn điểm đầu `i`, rồi với mỗi `i` chọn điểm cuối `j` từ `i` trở đi:

```python
for i in range(n):          # i = 0, 1, ..., n-1
    for j in range(i, n):   # j = i, i+1, ..., n-1   (j bắt đầu từ i, nên luôn i <= j)
        ...
```

Với `n = 4`, số đoạn là `4 + 3 + 2 + 1 = 10`. Tổng quát: `n + (n-1) + ... + 1 = n(n+1)/2` đoạn — tức khoảng `n²/2`.

In [4]:
n = 4
count = 0
for i in range(n):
    pairs = []
    for j in range(i, n):
        pairs.append((i, j))
        count += 1
    print(f"i = {i}: {pairs}")
print("Tổng số đoạn:", count, "| công thức n(n+1)/2 =", n * (n + 1) // 2)

i = 0: [(0, 0), (0, 1), (0, 2), (0, 3)]
i = 1: [(1, 1), (1, 2), (1, 3)]
i = 2: [(2, 2), (2, 3)]
i = 3: [(3, 3)]
Tổng số đoạn: 10 | công thức n(n+1)/2 = 10


### Cách 1a — Bản ngây thơ `O(n³)`

Người mới thường viết ngay: với mỗi cặp `(i, j)`, cắt `heights[i:j+1]` rồi gọi `min(...)`.

```text
best = 0
với mọi i <= j:
    best = max(best, min(heights[i..j]) × (j - i + 1))
```

Code rất ngắn và **gần như chắc chắn đúng** vì bám sát định nghĩa — đây là "thước đo chuẩn" để kiểm tra các cách nhanh hơn ở phía sau.

Nhưng có một cái giá ẩn: `min(heights[i:j+1])` phải **duyệt hết** `j - i + 1` phần tử (và slicing còn tạo một list mới cùng độ dài). Có `~n²/2` đoạn, mỗi đoạn tốn tới `n` bước → tổng cộng khoảng `n³/6` bước: **`O(n³)`**.

In [5]:
def largest_rectangle_naive(heights: List[int]) -> int:
    n = len(heights)
    best = 0
    for i in range(n):
        for j in range(i, n):
            h = min(heights[i:j + 1])            # duyệt lại cả đoạn -> tốn j-i+1 bước
            best = max(best, h * (j - i + 1))
    return best

print(largest_rectangle_naive(heights1))   # 10
print(largest_rectangle_naive(heights2))   # 4

10
4


### Cách 1b — "Min chạy" (running min): từ `O(n³)` xuống `O(n²)`

**Cách 1a vướng ở đâu?** Khi `i` cố định và `j` tăng dần, đoạn `i..j+1` chỉ hơn đoạn `i..j` đúng **một cột** (cột `j+1`). Vậy mà ta lại tính `min` của cả đoạn từ đầu. Công việc bị lặp!

**Mấu chốt:** `min` của đoạn dài hơn một phần tử = `min(min cũ, phần tử mới)`:

```text
min(heights[i..j+1]) = min( min(heights[i..j]), heights[j+1] )
```

Ví dụ `i = 2` trên `[2,1,5,6,2,3]`:

| j | cột mới `heights[j]` | min chạy | rộng | diện tích |
|---:|---:|---:|---:|---:|
| 2 | 5 | 5 | 1 | 5 |
| 3 | 6 | min(5, 6) = 5 | 2 | **10** |
| 4 | 2 | min(5, 2) = 2 | 3 | 6 |
| 5 | 3 | min(2, 3) = 2 | 4 | 8 |

Mỗi bước `j` chỉ tốn `O(1)` thay vì `O(n)`.

In [6]:
i = 2
cur_min = heights1[i]
for j in range(i, len(heights1)):
    cur_min = min(cur_min, heights1[j])     # cập nhật min chạy, KHÔNG duyệt lại
    print(f"j = {j}: heights[j] = {heights1[j]}, min chạy = {cur_min}, "
          f"rộng = {j - i + 1}, diện tích = {cur_min * (j - i + 1)}")

j = 2: heights[j] = 5, min chạy = 5, rộng = 1, diện tích = 5
j = 3: heights[j] = 6, min chạy = 5, rộng = 2, diện tích = 10
j = 4: heights[j] = 2, min chạy = 2, rộng = 3, diện tích = 6
j = 5: heights[j] = 3, min chạy = 2, rộng = 4, diện tích = 8


In [7]:
def largest_rectangle_bruteforce(heights: List[int]) -> int:
    n = len(heights)
    best = 0
    for i in range(n):
        cur_min = heights[i]
        for j in range(i, n):
            cur_min = min(cur_min, heights[j])          # O(1) mỗi bước
            best = max(best, cur_min * (j - i + 1))
    return best

print(largest_rectangle_bruteforce(heights1))   # 10
print(largest_rectangle_bruteforce(heights2))   # 4
print(largest_rectangle_bruteforce([2, 1, 2]))  # 3: cao 1, phủ cả 3 cột

10
4
3


### Trace cách 1b trên example 1

Hàm dưới in **mọi** đoạn `(i, j)` (21 đoạn với `n = 6`) kèm min chạy, diện tích và đánh dấu khi kỷ lục `best` được làm mới. Để ý:

- kỷ lục đổi ở `(0,0)` → 2, `(0,5)` → 6, và cuối cùng `(2,3)` → **10**;
- với `i = 1` (cột cao 1), min chạy kẹt ở 1 suốt — "cột thấp kéo cả đoạn xuống".

In [8]:
def trace_bruteforce(heights):
    n = len(heights)
    best = 0
    for i in range(n):
        cur_min = heights[i]
        for j in range(i, n):
            cur_min = min(cur_min, heights[j])
            area = cur_min * (j - i + 1)
            mark = ""
            if area > best:
                best = area
                mark = "  <- best mới"
            print(f"(i={i}, j={j}) min={cur_min} rộng={j - i + 1} area={area:>2} best={best:>2}{mark}")
    return best

_ = trace_bruteforce(heights1)

(i=0, j=0) min=2 rộng=1 area= 2 best= 2  <- best mới
(i=0, j=1) min=1 rộng=2 area= 2 best= 2
(i=0, j=2) min=1 rộng=3 area= 3 best= 3  <- best mới
(i=0, j=3) min=1 rộng=4 area= 4 best= 4  <- best mới
(i=0, j=4) min=1 rộng=5 area= 5 best= 5  <- best mới
(i=0, j=5) min=1 rộng=6 area= 6 best= 6  <- best mới
(i=1, j=1) min=1 rộng=1 area= 1 best= 6
(i=1, j=2) min=1 rộng=2 area= 2 best= 6
(i=1, j=3) min=1 rộng=3 area= 3 best= 6
(i=1, j=4) min=1 rộng=4 area= 4 best= 6
(i=1, j=5) min=1 rộng=5 area= 5 best= 6
(i=2, j=2) min=5 rộng=1 area= 5 best= 6
(i=2, j=3) min=5 rộng=2 area=10 best=10  <- best mới
(i=2, j=4) min=2 rộng=3 area= 6 best=10
(i=2, j=5) min=2 rộng=4 area= 8 best=10
(i=3, j=3) min=6 rộng=1 area= 6 best=10
(i=3, j=4) min=2 rộng=2 area= 4 best=10
(i=3, j=5) min=2 rộng=3 area= 6 best=10
(i=4, j=4) min=2 rộng=1 area= 2 best=10
(i=4, j=5) min=2 rộng=2 area= 4 best=10
(i=5, j=5) min=3 rộng=1 area= 3 best=10


### Cách 1 vướng ở đâu?

- **Time:** `O(n²)` với `n = len(heights)` — có `n(n+1)/2` đoạn, mỗi đoạn `O(1)`. (Bản 1a là `O(n³)`.)
- **Space:** `O(1)` (bản 1b chỉ dùng vài biến).

Với `n = 10^5`: khoảng `5 × 10^9` đoạn. Python làm được cỡ `10^7` phép đơn giản mỗi giây → mất vài trăm giây → **TLE**.

**Nút thắt:** ta thử **mọi** đoạn, nhưng phần lớn chúng **vô ích**. Nhìn lại bảng ở mục 3: đoạn `2..4` có min = 2, nhưng đoạn `2..5` cũng có min = 2 mà rộng hơn — đoạn `2..4` không bao giờ có thể là đáp án. Với **mỗi chiều cao**, chỉ có đoạn **rộng nhất** là đáng xét. Đó là ý tưởng của cách 2.

In [9]:
for n in [6, 1_000, 100_000]:
    print(f"n = {n:>7,}: số đoạn (i, j) = {n * (n + 1) // 2:>14,}")

n =       6: số đoạn (i, j) =             21
n =   1,000: số đoạn (i, j) =        500,500
n = 100,000: số đoạn (i, j) =  5,000,050,000


## 5. Cách 2 — Đổi góc nhìn: mỗi cột làm "chiều cao", lan sang trái/phải

### Insight: hình chữ nhật tốt nhất luôn **cao đúng bằng một cột nào đó**

Lấy một hình chữ nhật tối ưu, phủ đoạn `i..j`. Chiều cao của nó phải là `min(heights[i..j])`: nếu thấp hơn, ta nâng lên được, diện tích tăng → mâu thuẫn với "tối ưu". Mà `min` của đoạn chính là chiều cao của **một cột cụ thể** `k` trong đoạn — gọi là **cột thấp nhất** (cột "chặn trần").

Vậy thay vì duyệt **đoạn**, ta duyệt **cột**:

> Với mỗi cột `k`, giả sử hình chữ nhật cao **đúng** `heights[k]`. Nó được rộng tới đâu?
> Lan sang **trái** chừng nào các cột còn **cao ≥ `heights[k]`**, lan sang **phải** cũng vậy. Gặp cột **thấp hơn** thì dừng.

Ví dụ `k = 2` (cao 5) trên `[2,1,5,6,2,3]`:

- lan trái: cột 1 cao 1 < 5 → dừng ngay, `left = 2`;
- lan phải: cột 3 cao 6 ≥ 5 → đi tiếp; cột 4 cao 2 < 5 → dừng, `right = 3`;
- diện tích `5 × (3 - 2 + 1) = 10`.

Chỉ có `n` cột → `n` "ứng viên", thay vì `n²/2` đoạn. (Mỗi lần lan vẫn có thể tốn nhiều bước — xem ở cuối mục.)

### Kiến thức nền: `while` với điều kiện `and` và cái bẫy **chỉ số âm** của Python

Để lan sang trái ta viết:

```python
while left - 1 >= 0 and heights[left - 1] >= h:
    left -= 1
```

Hai điều quan trọng:

1. **`and` ngắn mạch (short-circuit):** Python tính vế trái trước; nếu vế trái là `False` thì **không tính** vế phải nữa. Nhờ vậy khi `left - 1 = -1`, ta không bao giờ chạm tới `heights[-1]`.
2. **Vì sao phải chặn `left - 1 >= 0`?** Trong Python, `heights[-1]` **không báo lỗi** mà trả về **phần tử cuối**! Quên điều kiện này, code sẽ lặng lẽ "lan vòng" sang cuối mảng và cho kết quả sai (xem phần lỗi thường gặp).

In [10]:
a = [2, 1, 5, 6, 2, 3]
print("a[-1] =", a[-1], "(phần tử CUỐI, không phải lỗi!)")
print("a[-2] =", a[-2])

x = -1
print("x >= 0 and a[x] > 100 ->", x >= 0 and a[x] > 100)   # vế trái False -> không đọc a[x]

a[-1] = 3 (phần tử CUỐI, không phải lỗi!)
a[-2] = 2
x >= 0 and a[x] > 100 -> False


In [11]:
def expand_from(heights, k):
    # Trả về (left, right): đoạn rộng nhất chứa k mà mọi cột đều cao >= heights[k]
    h = heights[k]
    left = k
    while left - 1 >= 0 and heights[left - 1] >= h:
        left -= 1
    right = k
    while right + 1 < len(heights) and heights[right + 1] >= h:
        right += 1
    return left, right

left, right = expand_from(heights1, 2)
print(f"k = 2: left = {left}, right = {right}, area = {heights1[2] * (right - left + 1)}")
draw_histogram(heights1, rect=(left, right, heights1[2]))

k = 2: left = 2, right = 3, area = 10
 6 |      ##    
 5 |    @@@@    
 4 |    @@@@    
 3 |    @@@@  ##
 2 |##  @@@@####
 1 |####@@@@####
   +------------
    0 1 2 3 4 5 


### Bảng lan cho từng cột của example 1

| k | `heights[k]` | lan trái tới `left` | lan phải tới `right` | rộng | diện tích |
|---:|---:|---:|---:|---:|---:|
| 0 | 2 | 0 (không có cột bên trái) | 0 (cột 1 cao 1 < 2) | 1 | 2 |
| 1 | 1 | 0 | 5 (mọi cột đều ≥ 1) | 6 | 6 |
| 2 | 5 | 2 (cột 1 cao 1 < 5) | 3 (cột 4 cao 2 < 5) | 2 | **10** |
| 3 | 6 | 3 | 3 | 1 | 6 |
| 4 | 2 | 2 (cột 3, 2 đều ≥ 2; cột 1 < 2) | 5 | 4 | 8 |
| 5 | 3 | 5 | 5 | 1 | 3 |

Max = **10** ở `k = 2`. Cột `k = 4` cũng đáng chú ý: cao chỉ 2 nhưng lan được tận 4 cột → 8.

In [12]:
def largest_rectangle_expand(heights: List[int]) -> int:
    best = 0
    for k in range(len(heights)):
        left, right = expand_from(heights, k)
        best = max(best, heights[k] * (right - left + 1))
    return best

def trace_expand(heights):
    best = 0
    for k in range(len(heights)):
        left, right = expand_from(heights, k)
        area = heights[k] * (right - left + 1)
        best = max(best, area)
        print(f"k={k} h={heights[k]}: lan tới [{left}..{right}], rộng={right - left + 1}, "
              f"area={area:>2}, best={best}")
    return best

_ = trace_expand(heights1)
print(largest_rectangle_expand(heights2))   # 4

k=0 h=2: lan tới [0..0], rộng=1, area= 2, best=2
k=1 h=1: lan tới [0..5], rộng=6, area= 6, best=6
k=2 h=5: lan tới [2..3], rộng=2, area=10, best=10
k=3 h=6: lan tới [3..3], rộng=1, area= 6, best=10
k=4 h=2: lan tới [2..5], rộng=4, area= 8, best=10
k=5 h=3: lan tới [5..5], rộng=1, area= 3, best=10
4


### Cách 2 vướng ở đâu?

Số "ứng viên" giảm từ `n²/2` xuống `n`, **nhưng** mỗi lần lan có thể đi rất xa. Worst case: mọi cột **bằng nhau**, ví dụ `[3, 3, 3, ..., 3]` — cột nào cũng lan hết cả mảng → `n` cột × `n` bước = **`O(n²)`**. Vẫn TLE với `n = 10^5`.

- **Time:** `O(n²)` worst case (trên dữ liệu ngẫu nhiên thường nhanh hơn nhiều).
- **Space:** `O(1)`.

Cell dưới đếm tổng số bước lan để thấy khác biệt.

**Nút thắt mới:** việc thật sự cần cho mỗi cột `k` là trả lời hai câu hỏi **"cột thấp hơn gần nhất bên trái / bên phải là ai?"**. Ta đang trả lời bằng cách đi bộ từng bước, và các cột cạnh nhau đi bộ **lại cùng một quãng đường**. Ta sẽ quay lại nút thắt này ở mục 7; trước đó, xem thêm một hướng nghĩ khác rất tự nhiên: chia để trị.

In [13]:
import random

def count_steps_expand(heights):
    steps = 0
    n = len(heights)
    for k in range(n):
        h = heights[k]
        left = k
        while left - 1 >= 0 and heights[left - 1] >= h:
            left -= 1
            steps += 1
        right = k
        while right + 1 < n and heights[right + 1] >= h:
            right += 1
            steps += 1
    return steps

random.seed(0)
n = 2000
print("n =", n)
print("  mảng ngẫu nhiên   :", f"{count_steps_expand([random.randint(0, 10**4) for _ in range(n)]):>9,} bước lan")
print("  mọi cột bằng nhau :", f"{count_steps_expand([3] * n):>9,} bước lan  (~ n² = {n * n:,})")

n = 2000
  mảng ngẫu nhiên   :    26,227 bước lan


  mọi cột bằng nhau : 3,998,000 bước lan  (~ n² = 4,000,000)


## 6. Cách 3 — Chia để trị (divide and conquer) theo cột thấp nhất

### Kiến thức nền: đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mỗi hàm đệ quy cần hai phần:

1. **Base case** (trường hợp cơ sở): bài toán đủ nhỏ để trả lời ngay, **không** gọi đệ quy nữa — nếu thiếu, hàm gọi mãi không dừng.
2. **Bước đệ quy**: chia bài lớn thành bài nhỏ hơn, gọi lại hàm, rồi ghép kết quả.

Ví dụ mini: tổng của đoạn `a[lo..hi]` = `a[lo]` + tổng của đoạn `a[lo+1..hi]`; đoạn rỗng (`lo > hi`) có tổng 0. Cell dưới in ra mỗi lần gọi, **thụt lề** theo độ sâu để bạn thấy "chồng" lời gọi (call stack).

In [14]:
def range_sum(a, lo, hi, depth=0):
    print("  " * depth + f"range_sum({lo}, {hi})")
    if lo > hi:                                   # base case: đoạn rỗng
        return 0
    return a[lo] + range_sum(a, lo + 1, hi, depth + 1)   # bài nhỏ hơn: bỏ phần tử đầu

print("Kết quả:", range_sum([4, 1, 7], 0, 2))

range_sum(0, 2)
  range_sum(1, 2)
    range_sum(2, 2)
      range_sum(3, 2)
Kết quả: 12


### Ý tưởng chia để trị cho histogram

Xét đoạn cột `lo..hi`. Gọi `m` là index của **cột thấp nhất** trong đoạn. Hình chữ nhật tốt nhất trong đoạn chỉ có thể rơi vào **một trong ba** trường hợp:

1. **Đi ngang qua cột `m`**: khi đó nó bị cột `m` chặn trần, nên cao tối đa `heights[m]` — và vì `m` thấp nhất, ta kéo nó rộng **hết cả đoạn**: diện tích `heights[m] × (hi - lo + 1)`.
2. **Nằm hẳn bên trái** `m`: tức nằm trong đoạn `lo..m-1` → gọi đệ quy.
3. **Nằm hẳn bên phải** `m`: tức nằm trong đoạn `m+1..hi` → gọi đệ quy.

Đáp án = `max` của ba số đó. Base case: đoạn rỗng (`lo > hi`) cho 0.

Với `[2,1,5,6,2,3]`:

```text
solve(0..5): min = 1 tại cột 1 -> cắt ngang 1×6 = 6
├── solve(0..0): min = 2 tại cột 0 -> 2×1 = 2
└── solve(2..5): min = 2 tại cột 4 -> 2×4 = 8
    ├── solve(2..3): min = 5 tại cột 2 -> 5×2 = 10   <- đáp án
    │   └── solve(3..3): 6×1 = 6
    └── solve(5..5): 3×1 = 3
```

Mỗi cột thấp nhất "cắt" histogram thành hai nửa — giống cột cao 0 cắt đôi histogram mà ta nhắc ở mục 1.

Ghi chú cú pháp: code dưới định nghĩa hàm `solve` **bên trong** hàm chính (nested function). `solve` nhìn thấy được biến `heights` của hàm bao ngoài nên không cần truyền lại mỗi lần gọi.

In [15]:
def index_of_min(heights, lo, hi):
    m = lo
    for k in range(lo + 1, hi + 1):
        if heights[k] < heights[m]:
            m = k
    return m

def largest_rectangle_divide_conquer(heights: List[int]) -> int:
    def solve(lo, hi):
        if lo > hi:                                # base case: đoạn rỗng
            return 0
        m = index_of_min(heights, lo, hi)          # O(hi - lo + 1)
        across = heights[m] * (hi - lo + 1)        # trường hợp 1: đi ngang qua m
        return max(across, solve(lo, m - 1), solve(m + 1, hi))
    return solve(0, len(heights) - 1)

print(largest_rectangle_divide_conquer(heights1))   # 10
print(largest_rectangle_divide_conquer(heights2))   # 4

10
4


In [16]:
def trace_divide_conquer(heights):
    def solve(lo, hi, depth):
        if lo > hi:
            return 0
        m = index_of_min(heights, lo, hi)
        across = heights[m] * (hi - lo + 1)
        print("    " * depth + f"solve({lo}..{hi}) {heights[lo:hi + 1]}: "
              f"min={heights[m]} tại cột {m} -> cắt ngang {heights[m]}×{hi - lo + 1} = {across}")
        best = max(across, solve(lo, m - 1, depth + 1), solve(m + 1, hi, depth + 1))
        print("    " * depth + f"=> solve({lo}..{hi}) trả về {best}")
        return best
    return solve(0, len(heights) - 1, 0)

print("Kết quả:", trace_divide_conquer(heights1))

solve(0..5) [2, 1, 5, 6, 2, 3]: min=1 tại cột 1 -> cắt ngang 1×6 = 6
    solve(0..0) [2]: min=2 tại cột 0 -> cắt ngang 2×1 = 2
    => solve(0..0) trả về 2
    solve(2..5) [5, 6, 2, 3]: min=2 tại cột 4 -> cắt ngang 2×4 = 8
        solve(2..3) [5, 6]: min=5 tại cột 2 -> cắt ngang 5×2 = 10
            solve(3..3) [6]: min=6 tại cột 3 -> cắt ngang 6×1 = 6
            => solve(3..3) trả về 6
        => solve(2..3) trả về 10
        solve(5..5) [3]: min=3 tại cột 5 -> cắt ngang 3×1 = 3
        => solve(5..5) trả về 3
    => solve(2..5) trả về 10
=> solve(0..5) trả về 10
Kết quả: 10


### Độ phức tạp cách 3 — đẹp khi cắt cân, tệ khi mảng đã sắp xếp

Mỗi lời gọi `solve(lo, hi)` tốn `O(hi - lo + 1)` để tìm min.

- **Khi cột thấp nhất thường nằm gần giữa** (cắt cân, như dữ liệu ngẫu nhiên): giống merge sort, mỗi "tầng" của cây đệ quy tốn tổng `O(n)`, có khoảng `log n` tầng → **`O(n log n)`** (trung bình).
- **Worst case: mảng tăng dần** `[1, 2, 3, ..., n]` (hoặc giảm dần). Cột thấp nhất luôn ở **mép** đoạn, mỗi lần chỉ "gọt" được 1 cột: `n + (n-1) + ... + 1` → **`O(n²)`**.
- **Space:** độ sâu đệ quy — `O(log n)` khi cân, nhưng **`O(n)`** ở worst case. Python mặc định chỉ cho khoảng 1000–3000 tầng đệ quy (`sys.getrecursionlimit()`), nên với `n = 10^5` mảng tăng dần sẽ gặp **`RecursionError`** chứ chưa nói tới TLE.

(Ghi chú nâng cao: dùng **segment tree** để tìm min của đoạn trong `O(log n)` sẽ đưa time về `O(n log n)` ở mọi trường hợp, nhưng không chữa được độ sâu đệ quy. Ta không cần đi đường vòng đó — cách 4, 5 vừa ngắn vừa `O(n)`.)

In [17]:
import sys

def count_work_dc(heights):
    work = 0
    def solve(lo, hi):
        nonlocal work                 # cho phép hàm con sửa biến work của hàm ngoài
        if lo > hi:
            return 0
        work += hi - lo + 1           # chi phí tìm min trên đoạn
        m = index_of_min(heights, lo, hi)
        return max(heights[m] * (hi - lo + 1), solve(lo, m - 1), solve(m + 1, hi))
    solve(0, len(heights) - 1)
    return work

random.seed(1)
n = 800
print(f"n = {n}: ngẫu nhiên -> {count_work_dc([random.randint(0, 10**4) for _ in range(n)]):>7,} bước tìm min")
print(f"n = {n}: tăng dần   -> {count_work_dc(list(range(1, n + 1))):>7,} bước tìm min (~ n²/2 = {n * n // 2:,})")

limit = sys.getrecursionlimit()
big_sorted = list(range(1, limit + 500))     # độ sâu đệ quy cần > giới hạn
try:
    largest_rectangle_divide_conquer(big_sorted)
    print("Chạy được (không ngờ!)")
except RecursionError:
    print(f"Mảng tăng dần dài {len(big_sorted)} -> RecursionError (giới hạn đệ quy = {limit})")

n = 800: ngẫu nhiên ->   9,264 bước tìm min
n = 800: tăng dần   -> 320,400 bước tìm min (~ n²/2 = 320,000)
Mảng tăng dần dài 1499 -> RecursionError (giới hạn đệ quy = 1000)


## 7. Bước đột phá: mọi thứ quy về "cột **thấp hơn gần nhất**"

Quay lại cách 2. Với cột `k`, ta lan trái tới khi gặp cột **thấp hơn** và lan phải tới khi gặp cột **thấp hơn**. Hãy đặt tên cho hai cột "chặn" đó:

- `L[k]` = index của cột **gần nhất bên trái** có chiều cao **nhỏ hơn hẳn** `heights[k]` (nếu không có, quy ước `L[k] = -1` — một "bức tường ảo" ngay trước cột 0);
- `R[k]` = index của cột **gần nhất bên phải** có chiều cao **nhỏ hơn hẳn** `heights[k]` (nếu không có, `R[k] = n` — bức tường ảo ngay sau cột cuối).

Hình chữ nhật cao `heights[k]` phủ đúng các cột **nằm giữa** `L[k]` và `R[k]` (không tính hai cột chặn):

```text
width[k] = R[k] - L[k] - 1
area[k]  = heights[k] × width[k]
```

**Vì sao `- 1`?** Các cột nằm giữa là `L+1, L+2, ..., R-1`. Đếm: `(R - 1) - (L + 1) + 1 = R - L - 1`. Ví dụ `k = 2`: `L = 1`, `R = 4` → các cột giữa là `2, 3` → rộng `4 - 1 - 1 = 2`. Ví dụ `k = 1`: `L = -1`, `R = 6` → rộng `6 - (-1) - 1 = 6`, đúng là cả mảng — bức tường ảo `-1` và `n` làm công thức chạy đều cho cả hai mép.

| k | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| `heights[k]` | 2 | 1 | 5 | 6 | 2 | 3 |
| `L[k]` (thấp hơn gần nhất bên trái) | -1 | -1 | 1 | 2 | 1 | 4 |
| `R[k]` (thấp hơn gần nhất bên phải) | 1 | 6 | 4 | 4 | 6 | 6 |
| `width = R - L - 1` | 1 | 6 | 2 | 1 | 4 | 1 |
| `area` | 2 | 6 | **10** | 6 | 8 | 3 |

Khớp đúng bảng lan ở mục 5 (`left = L + 1`, `right = R - 1`). Bài toán giờ chỉ còn: **tính `L[]` và `R[]` thật nhanh**. Đây là bài kinh điển **"Previous / Next Smaller Element"** — và công cụ chuẩn cho nó là **monotonic stack**.

In [18]:
def nearest_smaller_bruteforce(heights):
    # Cách chậm O(n²) chỉ để kiểm tra bảng ở trên
    n = len(heights)
    L, R = [-1] * n, [n] * n
    for k in range(n):
        for j in range(k - 1, -1, -1):       # đi lùi sang trái
            if heights[j] < heights[k]:
                L[k] = j
                break
        for j in range(k + 1, n):            # đi sang phải
            if heights[j] < heights[k]:
                R[k] = j
                break
    return L, R

L, R = nearest_smaller_bruteforce(heights1)
print("L     =", L)
print("R     =", R)
print("width =", [R[k] - L[k] - 1 for k in range(len(heights1))])
print("area  =", [heights1[k] * (R[k] - L[k] - 1) for k in range(len(heights1))])

L     = [-1, -1, 1, 2, 1, 4]
R     = [1, 6, 4, 4, 6, 6]
width = [1, 6, 2, 1, 4, 1]
area  = [2, 6, 10, 6, 8, 3]


### Kiến thức nền: stack (ngăn xếp)

**Stack** giống một chồng đĩa: đĩa đặt vào **sau cùng** được lấy ra **đầu tiên** — gọi là **LIFO** (Last In, First Out). Trong Python, dùng luôn `list`:

| Thao tác | Code | Ý nghĩa | Chi phí |
|---|---|---|---|
| push (đặt lên đỉnh) | `stack.append(x)` | thêm vào cuối list | `O(1)` |
| pop (lấy khỏi đỉnh) | `stack.pop()` | bỏ và trả về phần tử cuối | `O(1)` |
| peek (nhìn đỉnh) | `stack[-1]` | xem phần tử cuối, không bỏ | `O(1)` |
| rỗng chưa? | `if not stack:` | list rỗng thì `not stack` là `True` | `O(1)` |

(Ở đây `stack[-1]` dùng đúng tính năng "chỉ số âm" vừa học: phần tử cuối = đỉnh stack.)

In [19]:
stack = []
stack.append(10)
stack.append(20)
stack.append(30)
print("stack sau 3 lần push:", stack)
print("đỉnh stack[-1]     :", stack[-1])
print("pop ra             :", stack.pop())
print("stack còn lại      :", stack)
print("stack rỗng?        :", not stack)

stack sau 3 lần push: [10, 20, 30]
đỉnh stack[-1]     : 30
pop ra             : 30
stack còn lại      : [10, 20]
stack rỗng?        : False


### Monotonic stack để tìm `L[]` (cột thấp hơn gần nhất bên trái)

Duyệt `i` từ trái sang phải, giữ một stack chứa **index**, sao cho chiều cao **tăng dần** từ đáy lên đỉnh (**monotonic increasing stack**). Tại mỗi `i`:

1. **Pop** mọi index ở đỉnh có chiều cao `>= heights[i]`.
2. Sau khi pop, đỉnh stack (nếu còn) chính là `L[i]`; stack rỗng thì `L[i] = -1`.
3. **Push** `i`.

**Vì sao pop đi là an toàn — không bao giờ cần lại?** Giả sử `j < i` và `heights[j] >= heights[i]`. Với mọi cột `k` đến sau (`k > i`), nếu `j` thấp hơn `k` thì `i` cũng thấp hơn `k` (vì `heights[i] <= heights[j]`), mà `i` lại **gần `k` hơn**. Vậy `j` **không bao giờ** là "thấp hơn gần nhất" của ai nữa — nó bị `i` "che khuất" vĩnh viễn. Bỏ nó đi không mất gì.

Trace trên `[2,1,5,6,2,3]`:

| i | `heights[i]` | pop (vì cao ≥ `heights[i]`) | `L[i]` = đỉnh sau khi pop | stack sau push |
|---:|---:|---|---:|---|
| 0 | 2 | — | -1 (rỗng) | `[0]` |
| 1 | 1 | 0 (h=2) | -1 (rỗng) | `[1]` |
| 2 | 5 | — | 1 | `[1, 2]` |
| 3 | 6 | — | 2 | `[1, 2, 3]` |
| 4 | 2 | 3 (h=6), 2 (h=5) | 1 | `[1, 4]` |
| 5 | 3 | — | 4 | `[1, 4, 5]` |

Kết quả `L = [-1, -1, 1, 2, 1, 4]` — khớp bảng mục 7.

Cú pháp mới: `x if dieu_kien else y` là **biểu thức điều kiện** — trả về `x` khi điều kiện đúng, ngược lại trả về `y`. Ví dụ `stack[-1] if stack else -1`.

In [20]:
def previous_smaller(heights, verbose=False):
    n = len(heights)
    left = [-1] * n
    stack = []                                   # index, chiều cao tăng dần từ đáy lên đỉnh
    for i in range(n):
        popped = []
        while stack and heights[stack[-1]] >= heights[i]:
            popped.append(stack.pop())
        left[i] = stack[-1] if stack else -1
        stack.append(i)
        if verbose:
            print(f"i={i} h={heights[i]}: pop {popped}, L[{i}] = {left[i]:>2}, stack = {stack}")
    return left

print("L =", previous_smaller(heights1, verbose=True))

i=0 h=2: pop [], L[0] = -1, stack = [0]
i=1 h=1: pop [0], L[1] = -1, stack = [1]
i=2 h=5: pop [], L[2] =  1, stack = [1, 2]
i=3 h=6: pop [], L[3] =  2, stack = [1, 2, 3]
i=4 h=2: pop [3, 2], L[4] =  1, stack = [1, 4]
i=5 h=3: pop [], L[5] =  4, stack = [1, 4, 5]
L = [-1, -1, 1, 2, 1, 4]


### Lượt thứ hai: `R[]` — làm y hệt nhưng duyệt **từ phải sang trái**

"Thấp hơn gần nhất bên phải" chính là "thấp hơn gần nhất bên trái" nếu ta **đọc mảng ngược**. Ta duyệt `i = n-1, n-2, ..., 0` bằng `range(n - 1, -1, -1)`:

- `range(start, stop, step)` dừng **trước** `stop`; với `step = -1` thì đi lùi. Muốn tới index 0 thì `stop` phải là `-1`.
- Stack rỗng → `R[i] = n` (bức tường ảo bên phải).

In [21]:
print("range(5, -1, -1) ->", list(range(5, -1, -1)))

def next_smaller(heights):
    n = len(heights)
    right = [n] * n
    stack = []
    for i in range(n - 1, -1, -1):               # i = n-1, ..., 0
        while stack and heights[stack[-1]] >= heights[i]:
            stack.pop()
        right[i] = stack[-1] if stack else n
        stack.append(i)
    return right

print("R =", next_smaller(heights1))             # mong đợi [1, 6, 4, 4, 6, 6]

range(5, -1, -1) -> [5, 4, 3, 2, 1, 0]
R = [1, 6, 4, 4, 6, 6]


## 8. Cách 4 — Hai mảng `left_smaller` / `right_smaller` bằng monotonic stack

Ghép lại:

1. Lượt trái → phải: `left_smaller = previous_smaller(heights)`.
2. Lượt phải → trái: `right_smaller = next_smaller(heights)`.
3. Lượt cuối: `best = max(heights[k] × (right_smaller[k] - left_smaller[k] - 1))`.

Công thức giống hệt cách 2; ta chỉ đổi cách **tìm biên** từ "đi bộ từng bước" sang "tra bảng đã tính sẵn bằng stack".

**Về cột bằng nhau:** cả hai lượt đều pop khi `>=`, nên cả `L` lẫn `R` đều là cột **nhỏ hơn hẳn**. Nhờ vậy mỗi cột được lan **rộng tối đa** — với `[3, 3]`, cả hai cột đều được rộng 2, diện tích 6. (Nếu một lượt dùng `>` thì một trong các cột bằng nhau bị tính hụt, nhưng cột còn lại vẫn đủ rộng nên đáp án vẫn đúng; dùng `>=` cả hai phía thì dễ lý luận nhất.)

In [22]:
def largest_rectangle_two_pass(heights: List[int]) -> int:
    left_smaller = previous_smaller(heights)
    right_smaller = next_smaller(heights)
    best = 0
    for k in range(len(heights)):
        width = right_smaller[k] - left_smaller[k] - 1
        best = max(best, heights[k] * width)
    return best

print(largest_rectangle_two_pass(heights1))   # 10
print(largest_rectangle_two_pass(heights2))   # 4
print(largest_rectangle_two_pass([3, 3]))     # 6
print("L, R của [3, 3]:", previous_smaller([3, 3]), next_smaller([3, 3]))

10
4
6
L, R của [3, 3]: [-1, -1] [2, 2]


### Độ phức tạp cách 4

Vòng `while` nằm trong vòng `for` trông như `O(n²)`, nhưng hãy đếm theo **từng index**: mỗi index được **push đúng 1 lần** và **pop tối đa 1 lần** trong một lượt. Vậy cả lượt có tối đa `n` lần pop → tổng công việc `O(n)`. Kiểu phân tích "đếm tổng cả quá trình thay vì từng bước" này gọi là **amortized analysis** (phân tích khấu hao).

- **Time:** `O(n)` — ba lượt, mỗi lượt `O(n)`.
- **Space:** `O(n)` — hai mảng `left_smaller`, `right_smaller` và stack.

**Còn có thể gọn hơn không?** Nhìn kỹ lượt trái → phải: khi cột `j` bị **pop tại `i`**, đó là vì `heights[i] <= heights[j]` — `i` chính là cột đầu tiên bên phải **không cao hơn** `j`, tức (gần như) là `R[j]`! Và cột nằm ngay dưới `j` trong stack chính là `L[j]`. Nghĩa là **tại khoảnh khắc pop, ta đã biết đủ cả hai biên** — không cần lượt thứ hai. Cell dưới kiểm chứng: ghi lại "ai pop `j`" và so với `R`.

In [23]:
def popped_by(heights):
    n = len(heights)
    who = [n] * n                    # mặc định n: không ai pop (còn lại trong stack tới cuối)
    stack = []
    for i in range(n):
        while stack and heights[stack[-1]] >= heights[i]:
            who[stack.pop()] = i
        stack.append(i)
    return who

print("ai pop j  :", popped_by(heights1))
print("R (lượt 2):", next_smaller(heights1))

ai pop j  : [1, 6, 4, 4, 6, 6]
R (lượt 2): [1, 6, 4, 4, 6, 6]


## 9. Cách 5 — Monotonic stack **một lượt** với sentinel (cách chính)

### Thuật toán bằng lời

Duyệt `i` từ trái sang phải. Stack chứa **index** các cột "đang chờ" — những cột **chưa biết** mình rộng tới đâu vì chưa gặp cột thấp hơn ở bên phải. Chiều cao trong stack **không giảm** từ đáy lên đỉnh.

Tại mỗi `i` có chiều cao `h`:

1. **Chừng nào** cột ở đỉnh stack **cao hơn** `h`: cột đó (gọi là `top`) vừa tìm thấy **biên phải** là `i`. Pop nó ra; lúc này **đỉnh mới** `stack[-1]` là **biên trái** của nó. Tính
   `width = i - stack[-1] - 1`, `area = heights[top] × width`, cập nhật `best`.
2. Push `i`.

Có hai **sentinel** (lính canh — phần tử giả thêm vào để khỏi phải viết code xử lý trường hợp đặc biệt):

- **Sentinel trái `-1` ở đáy stack**: đóng vai "bức tường ảo" trước cột 0 (đúng như `L = -1` ở mục 7). Nhờ nó, sau khi pop **luôn** có `stack[-1]` để làm biên trái, và công thức `width = i - stack[-1] - 1` dùng được cho mọi trường hợp. Ta không bao giờ pop nó (điều kiện `stack[-1] != -1`).
- **Sentinel phải: một cột cao `0` ở vị trí `n`** (sau cột cuối). Vì `0` không cao hơn cột nào, khi tới đây mọi cột cao hơn 0 còn "chờ" trong stack đều bị pop và được tính diện tích. Không có nó, các cột còn sót trong stack (ví dụ mảng tăng dần `[1,2,3,4,5]` không bao giờ pop!) sẽ bị bỏ quên.

### Vì sao `width = i - stack[-1] - 1`?

Ngay sau khi pop `top`:

- **biên phải** là `i`: `heights[i] < heights[top]`, và mọi cột giữa `top` và `i` đều cao ≥ `heights[top]` (nếu có cột nào thấp hơn, nó đã pop `top` từ trước rồi);
- **biên trái** là `L = stack[-1]` (đỉnh mới): cột thấp hơn-hoặc-bằng gần nhất bên trái, mọi cột giữa `L` và `top` đều cao hơn `heights[top]` — chúng chính là những cột đã bị pop **trước đó**, và **vẫn thuộc** hình chữ nhật của `top`.

Hình chữ nhật phủ các cột `L+1 .. i-1` → rộng `(i - 1) - (L + 1) + 1 = i - L - 1`.

**Đừng** nhầm thành `i - top`: các cột nằm giữa `L` và `top` có thể đã bị pop khỏi stack, nhưng hình chữ nhật của `top` vẫn **lan qua** chúng. Ví dụ ở bước `i = 6` bên dưới: pop cột 1 (cao 1), `L = -1`, rộng `6 - (-1) - 1 = 6` (cả mảng), trong khi `i - top = 5` thiếu mất cột 0.

### Kiến thức Python: `heights + [0]` (tạo list mới) khác `heights.append(0)` (sửa list cũ)

Để thêm sentinel `0` vào cuối, có hai cách và chúng **khác nhau**:

- `hs = heights + [0]` tạo **list mới**; `heights` của người gọi **giữ nguyên**. Tốn thêm `O(n)` bộ nhớ.
- `heights.append(0)` **sửa thẳng** list gốc — người gọi sẽ thấy list của họ bị dài thêm! Nếu dùng cách này, nhớ `heights.pop()` trả lại trước khi return.

Trong code nộp LeetCode ta dùng cách thứ ba gọn nhất: duyệt `i` tới `n` và coi `h = 0` khi `i == n`, không đụng gì vào list.

In [24]:
original = [2, 1, 5]
hs = original + [0]
print("hs =", hs, "| original =", original, "(không đổi)")

original.append(0)
print("sau append: original =", original, "(bị sửa!)")
original.pop()
print("sau pop   : original =", original)

hs = [2, 1, 5, 0] | original = [2, 1, 5] (không đổi)
sau append: original = [2, 1, 5, 0] (bị sửa!)
sau pop   : original = [2, 1, 5]


In [25]:
def largest_rectangle_stack(heights: List[int]) -> int:
    hs = heights + [0]               # sentinel phải: cột cao 0 ở vị trí n
    stack = [-1]                     # sentinel trái: bức tường ảo -1, không bao giờ bị pop
    best = 0
    for i, h in enumerate(hs):
        while stack[-1] != -1 and hs[stack[-1]] > h:
            top = stack.pop()                    # cột top vừa gặp biên phải i
            width = i - stack[-1] - 1            # biên trái = đỉnh mới
            best = max(best, hs[top] * width)
        stack.append(i)
    return best

print(largest_rectangle_stack(heights1))   # 10
print(largest_rectangle_stack(heights2))   # 4
print(largest_rectangle_stack([1, 2, 3, 4, 5]))   # 9 = 3×3 (cột 2..4), chỉ tìm được nhờ sentinel 0

10
4
9


### Trace cách 5 trên example 1 `[2,1,5,6,2,3]` (+ sentinel `0` ở `i = 6`)

Mỗi dòng là **một hành động**: pop (kèm tính diện tích) hoặc push. Cột "stack sau" là stack **ngay sau** hành động đó. `L` là đỉnh stack sau khi pop (biên trái).

| i | h | hành động | L | width = i - L - 1 | area | best | stack sau |
|---:|---|---|---:|---|---|---:|---|
| 0 | 2 | push `0` | | | | 0 | `[-1, 0]` |
| 1 | 1 | pop `0` (h=2) | -1 | 1 - (-1) - 1 = 1 | 2×1 = **2** | 2 | `[-1]` |
| 1 | 1 | push `1` | | | | 2 | `[-1, 1]` |
| 2 | 5 | push `2` | | | | 2 | `[-1, 1, 2]` |
| 3 | 6 | push `3` | | | | 2 | `[-1, 1, 2, 3]` |
| 4 | 2 | pop `3` (h=6) | 2 | 4 - 2 - 1 = 1 | 6×1 = **6** | 6 | `[-1, 1, 2]` |
| 4 | 2 | pop `2` (h=5) | 1 | 4 - 1 - 1 = 2 | 5×2 = **10** | 10 | `[-1, 1]` |
| 4 | 2 | push `4` | | | | 10 | `[-1, 1, 4]` |
| 5 | 3 | push `5` | | | | 10 | `[-1, 1, 4, 5]` |
| 6 | 0 (sentinel) | pop `5` (h=3) | 4 | 6 - 4 - 1 = 1 | 3×1 = **3** | 10 | `[-1, 1, 4]` |
| 6 | 0 (sentinel) | pop `4` (h=2) | 1 | 6 - 1 - 1 = 4 | 2×4 = **8** | 10 | `[-1, 1]` |
| 6 | 0 (sentinel) | pop `1` (h=1) | -1 | 6 - (-1) - 1 = 6 | 1×6 = **6** | 10 | `[-1]` |
| 6 | 0 (sentinel) | push `6` | | | | 10 | `[-1, 6]` |

Những điểm đáng chú ý:

- `i = 2, 3`: chiều cao tăng → chỉ push, chưa ai biết mình rộng bao nhiêu.
- `i = 4` (h = 2): **pop hai lần liên tiếp**. Cột 3 (h=6) rộng 1 → 6; rồi cột 2 (h=5): `L = 1`, rộng `4 - 1 - 1 = 2` → **10**. Cột 2 "mượn" luôn cột 3 vừa bị pop — đó là lý do phải tính theo `L`, không theo `top`.
- `i = 6` (sentinel 0): **dọn sạch** stack. Cột 4 (h=2) rộng `6 - 1 - 1 = 4` → 8; cột 1 (h=1) có `L = -1` → rộng 6 → 6.
- Mỗi index được push đúng 1 lần, pop đúng 1 lần (trừ sentinel).

In [26]:
def trace_stack(heights):
    hs = heights + [0]
    stack = [-1]
    best = 0
    for i, h in enumerate(hs):
        tag = " (sentinel)" if i == len(heights) else ""
        while stack[-1] != -1 and hs[stack[-1]] > h:
            top = stack.pop()
            L = stack[-1]
            width = i - L - 1
            area = hs[top] * width
            best = max(best, area)
            Ls = f"({L})" if L < 0 else str(L)      # in -1 trong ngoặc cho dễ đọc
            print(f"i={i} h={h}{tag}: pop {top} (h={hs[top]}), L={L:>2}, "
                  f"width={i}-{Ls}-1={width}, area={area:>2}, best={best:>2}, stack={stack}")
        stack.append(i)
        print(f"i={i} h={h}{tag}: push {i} -> stack={stack}")
    return best

print("Kết quả:", trace_stack(heights1))

i=0 h=2: push 0 -> stack=[-1, 0]
i=1 h=1: pop 0 (h=2), L=-1, width=1-(-1)-1=1, area= 2, best= 2, stack=[-1]
i=1 h=1: push 1 -> stack=[-1, 1]
i=2 h=5: push 2 -> stack=[-1, 1, 2]
i=3 h=6: push 3 -> stack=[-1, 1, 2, 3]
i=4 h=2: pop 3 (h=6), L= 2, width=4-2-1=1, area= 6, best= 6, stack=[-1, 1, 2]
i=4 h=2: pop 2 (h=5), L= 1, width=4-1-1=2, area=10, best=10, stack=[-1, 1]
i=4 h=2: push 4 -> stack=[-1, 1, 4]
i=5 h=3: push 5 -> stack=[-1, 1, 4, 5]
i=6 h=0 (sentinel): pop 5 (h=3), L= 4, width=6-4-1=1, area= 3, best=10, stack=[-1, 1, 4]
i=6 h=0 (sentinel): pop 4 (h=2), L= 1, width=6-1-1=4, area= 8, best=10, stack=[-1, 1]
i=6 h=0 (sentinel): pop 1 (h=1), L=-1, width=6-(-1)-1=6, area= 6, best=10, stack=[-1]
i=6 h=0 (sentinel): push 6 -> stack=[-1, 6]
Kết quả: 10


### Nhìn thấy các hình chữ nhật được tính khi pop

Mỗi lần pop sinh ra **một** hình chữ nhật ứng viên: phủ cột `L+1 .. i-1`, cao `heights[top]`. Cell dưới vẽ hai hình chữ nhật đáng chú ý nhất từ bảng trên:

- pop cột 2 tại `i = 4`: `L = 1` → phủ cột `2..3`, cao 5 → **10** (đáp án);
- pop cột 4 tại `i = 6`: `L = 1` → phủ cột `2..5`, cao 2 → 8 — để ý nó phủ cả cột 2, 3 đã bị pop từ trước.

In [27]:
def pop_rectangles(heights):
    # Trả về list (top, L, i, area) cho mỗi lần pop
    hs = heights + [0]
    stack, rects = [-1], []
    for i, h in enumerate(hs):
        while stack[-1] != -1 and hs[stack[-1]] > h:
            top = stack.pop()
            L = stack[-1]
            rects.append((top, L, i, hs[top] * (i - L - 1)))
        stack.append(i)
    return rects

for top, L, i, area in pop_rectangles(heights1):
    if top in (2, 4):
        print(f"pop cột {top} tại i={i}: phủ cột {L + 1}..{i - 1}, cao {heights1[top]}, area={area}")
        draw_histogram(heights1, rect=(L + 1, i - 1, heights1[top]))
        print()

pop cột 2 tại i=4: phủ cột 2..3, cao 5, area=10
 6 |      ##    
 5 |    @@@@    
 4 |    @@@@    
 3 |    @@@@  ##
 2 |##  @@@@####
 1 |####@@@@####
   +------------
    0 1 2 3 4 5 

pop cột 4 tại i=6: phủ cột 2..5, cao 2, area=8
 6 |      ##    
 5 |    ####    
 4 |    ####    
 3 |    ####  ##
 2 |##  @@@@@@@@
 1 |####@@@@@@@@
   +------------
    0 1 2 3 4 5 



### Trace thứ hai: cột **bằng nhau** `[1,3,3,2]`

Điều kiện pop là `>` (cao hơn **hẳn**), nên hai cột cao 3 cùng nằm trong stack. Xem điều gì xảy ra:

| i | h | hành động | L | width = i - L - 1 | area | best | stack sau |
|---:|---|---|---:|---|---|---:|---|
| 0 | 1 | push `0` | | | | 0 | `[-1, 0]` |
| 1 | 3 | push `1` | | | | 0 | `[-1, 0, 1]` |
| 2 | 3 | push `2` | | | | 0 | `[-1, 0, 1, 2]` |
| 3 | 2 | pop `2` (h=3) | 1 | 3 - 1 - 1 = 1 | 3×1 = **3** | 3 | `[-1, 0, 1]` |
| 3 | 2 | pop `1` (h=3) | 0 | 3 - 0 - 1 = 2 | 3×2 = **6** | 6 | `[-1, 0]` |
| 3 | 2 | push `3` | | | | 6 | `[-1, 0, 3]` |
| 4 | 0 (sentinel) | pop `3` (h=2) | 0 | 4 - 0 - 1 = 3 | 2×3 = **6** | 6 | `[-1, 0]` |
| 4 | 0 (sentinel) | pop `0` (h=1) | -1 | 4 - (-1) - 1 = 4 | 1×4 = **4** | 6 | `[-1]` |
| 4 | 0 (sentinel) | push `4` | | | | 6 | `[-1, 4]` |

Tại `i = 3`, cột 2 bị pop **trước** và chỉ được rộng 1 (vì đỉnh mới là cột 1 cùng cao 3, bị coi như "biên trái") — **tính hụt**! Nhưng ngay sau đó, trong **cùng vòng while**, cột 1 bị pop với cùng `i = 3` và được rộng đủ 2 → diện tích đúng `3 × 2 = 6`. Quy luật: trong một nhóm cột bằng nhau nằm liền nhau trong stack, cột **dưới cùng** luôn nhận đủ độ rộng, nên đáp án không bao giờ bị thiếu.

(Nếu đổi điều kiện thành `>=`, cột bằng nhau sẽ pop nhau ngay khi gặp; khi đó cột bị pop sớm tính hụt bên phải, còn cột cuối cùng của nhóm tính đủ. Cả hai cách đều đúng — chọn một và hiểu vì sao.)

In [28]:
print("Kết quả:", trace_stack([1, 3, 3, 2]))

i=0 h=1: push 0 -> stack=[-1, 0]
i=1 h=3: push 1 -> stack=[-1, 0, 1]
i=2 h=3: push 2 -> stack=[-1, 0, 1, 2]
i=3 h=2: pop 2 (h=3), L= 1, width=3-1-1=1, area= 3, best= 3, stack=[-1, 0, 1]
i=3 h=2: pop 1 (h=3), L= 0, width=3-0-1=2, area= 6, best= 6, stack=[-1, 0]
i=3 h=2: push 3 -> stack=[-1, 0, 3]
i=4 h=0 (sentinel): pop 3 (h=2), L= 0, width=4-0-1=3, area= 6, best= 6, stack=[-1, 0]
i=4 h=0 (sentinel): pop 0 (h=1), L=-1, width=4-(-1)-1=4, area= 4, best= 6, stack=[-1]
i=4 h=0 (sentinel): push 4 -> stack=[-1, 4]
Kết quả: 6


### Vì sao đúng? (invariant bằng lời)

Ở **đầu mỗi vòng** `i`:

1. Các index trong stack (trừ `-1`) **tăng dần theo vị trí** và chiều cao **không giảm** từ đáy lên đỉnh.
2. Mỗi cột `j < i` **đã bị pop** thì hình chữ nhật cao `heights[j]` rộng tối đa của nó (hoặc một hình cùng diện tích của cột bằng nó) **đã được tính** vào `best`.
3. Mỗi cột còn trong stack là cột **chưa gặp** cột thấp hơn ở bên phải — chưa biết biên phải nên **chưa được** tính.

Hai điều cần cho tính đúng:

- **Không đếm khống:** khi pop `top` tại `i`, mọi cột trong `L+1 .. i-1` đều cao ≥ `heights[top]`, nên hình chữ nhật đó **thật sự nằm trong** histogram.
- **Không bỏ sót:** hình chữ nhật tối ưu có một cột thấp nhất `k` (mục 5). Nhờ sentinel `0`, cột `k` chắc chắn bị pop, và (với cột thấp nhất "dưới cùng" trong nhóm bằng nhau) độ rộng được tính đúng bằng đoạn lan tối đa — tức diện tích tối ưu được xét tới.

Cell dưới **kiểm chứng bằng máy** cả hai ý trên tập mảng ngẫu nhiên: (a) mọi hình chữ nhật sinh ra khi pop đều hợp lệ (`min` của đoạn đúng bằng chiều cao), (b) `max` của chúng bằng đáp án brute force.

In [29]:
random.seed(84)
for _ in range(1000):
    h = [random.randint(0, 5) for _ in range(random.randint(1, 9))]
    rects = pop_rectangles(h)
    for top, L, i, area in rects:
        assert min(h[L + 1:i]) == h[top], (h, top, L, i)          # (a) không đếm khống
    assert max([0] + [a for *_, a in rects]) == largest_rectangle_bruteforce(h), h   # (b) không bỏ sót
print("1000 mảng ngẫu nhiên: mọi hình chữ nhật khi pop đều hợp lệ, và max = brute force")

1000 mảng ngẫu nhiên: mọi hình chữ nhật khi pop đều hợp lệ, và max = brute force


### Độ phức tạp cách 5

Gọi `n = len(heights)`.

- **Time:** `O(n)`. Mỗi index `0..n` được push **đúng 1 lần**; mỗi index `0..n-1` bị pop **tối đa 1 lần**. Tổng số lần chạy thân `while` ≤ `n` → tổng công việc `O(n)` (amortized, giống cách 4).
- **Space:** `O(n)` cho stack. Worst case là mảng **tăng dần** `[1, 2, ..., n]`: không ai bị pop cho tới sentinel, stack chứa đủ `n + 1` phần tử.

So với cách 4: cùng `O(n)` nhưng **một lượt duy nhất**, không cần hai mảng phụ — code ngắn, ít chỗ sai.

## 10. Code nộp LeetCode

Bản submit dùng cách 5. Thay vì `heights + [0]`, ta duyệt `i` từ `0` tới `n` (gồm `n`) và gán `h = 0` khi `i == n` — vẫn là sentinel `0` nhưng **không tạo list mới và không sửa input**. Code submit không `print`, đúng signature `largestRectangleArea(self, heights: List[int]) -> int`.

In [30]:
class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        n = len(heights)
        stack = [-1]                                   # sentinel trái
        best = 0
        for i in range(n + 1):
            h = heights[i] if i < n else 0             # sentinel phải: cột cao 0 tại i = n
            while stack[-1] != -1 and heights[stack[-1]] > h:
                top = stack.pop()
                width = i - stack[-1] - 1
                best = max(best, heights[top] * width)
            stack.append(i)
        return best


sol = Solution()
assert sol.largestRectangleArea([2, 1, 5, 6, 2, 3]) == 10      # example 1
assert sol.largestRectangleArea([2, 4]) == 4                   # example 2
assert sol.largestRectangleArea([0]) == 0                      # một cột cao 0
assert sol.largestRectangleArea([7]) == 7                      # một cột
assert sol.largestRectangleArea([0, 0, 0]) == 0                # toàn 0
assert sol.largestRectangleArea([1, 2, 3, 4, 5]) == 9          # tăng dần: cần sentinel phải
assert sol.largestRectangleArea([5, 4, 3, 2, 1]) == 9          # giảm dần
assert sol.largestRectangleArea([3, 3, 3, 3]) == 12            # bằng nhau
assert sol.largestRectangleArea([2, 1, 2]) == 3                # cột thấp ở giữa, rộng cả mảng
assert sol.largestRectangleArea([4, 0, 4]) == 4                # cột 0 cắt đôi histogram
assert sol.largestRectangleArea([6, 2, 5, 4, 5, 1, 6]) == 12   # 4 × 3 ở cột 2..4
data = [2, 1, 5]
sol.largestRectangleArea(data)
assert data == [2, 1, 5]                                       # input không bị sửa
print("Tất cả test Solution PASS")

Tất cả test Solution PASS


### Thử với input lớn nhất theo constraints

`n = 10^5`, chiều cao tới `10^4`. Cách 5 chạy trong tích tắc; còn brute force `O(n²)` thì đừng thử. Ta thử cả mảng tăng dần (worst case bộ nhớ của stack, và cũng là worst case của chia để trị) để thấy cách 5 không hề "sợ" nó.

In [31]:
import time

random.seed(7)
big_random = [random.randint(0, 10**4) for _ in range(10**5)]
big_increasing = [k // 10 for k in range(10**5)]   # 0,0,...,9999: tăng dần, chiều cao <= 10^4 đúng constraints

for name, arr in [("ngẫu nhiên", big_random), ("tăng dần", big_increasing), ("bằng nhau", [10**4] * 10**5)]:
    start = time.perf_counter()
    ans = Solution().largestRectangleArea(arr)
    elapsed = time.perf_counter() - start
    print(f"n = {len(arr):,} ({name:<10}): đáp án = {ans:>12,}, thời gian ~ {elapsed:.3f} giây")

n = 100,000 (ngẫu nhiên): đáp án =      130,590, thời gian ~ 0.023 giây


n = 100,000 (tăng dần  ): đáp án =  250,000,000, thời gian ~ 0.026 giây


n = 100,000 (bằng nhau ): đáp án = 1,000,000,000, thời gian ~ 0.025 giây


### Kiểm thử đối chiếu chéo tất cả các cách

Bản ngây thơ `O(n³)` chậm nhưng **dễ tin là đúng** vì bám sát định nghĩa. Ta sinh ngẫu nhiên nhiều mảng nhỏ (cố định `random.seed` để lần nào chạy cũng ra cùng bộ test), cố ý cho chiều cao trong khoảng nhỏ `0..6` để có **nhiều cột bằng nhau và nhiều cột 0** — đúng những chỗ dễ sai — rồi so mọi cách với nhau.

In [32]:
random.seed(2026)
methods = [
    largest_rectangle_bruteforce,
    largest_rectangle_expand,
    largest_rectangle_divide_conquer,
    largest_rectangle_two_pass,
    largest_rectangle_stack,
    Solution().largestRectangleArea,
]
for _ in range(2000):
    h = [random.randint(0, 6) for _ in range(random.randint(1, 10))]
    expected = largest_rectangle_naive(h)
    for f in methods:
        assert f(h) == expected, (getattr(f, "__name__", f), h, f(h), expected)
print(f"2000 mảng ngẫu nhiên: {len(methods)} cách đều khớp với bản ngây thơ O(n³)")

2000 mảng ngẫu nhiên: 6 cách đều khớp với bản ngây thơ O(n³)


## 11. So sánh các cách

`n = len(heights)`.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---|---|---|
| 1a. Brute force ngây thơ | Mọi đoạn `(i, j)`, tính `min` lại từ đầu | `O(n³)` | `O(n)` (slice) | Thước đo chuẩn để test |
| 1b. Brute force + min chạy | Mọi đoạn, `min` cập nhật dần | `O(n²)` | `O(1)` | TLE với `n = 10^5` |
| 2. Lan trái/phải | Mỗi cột làm chiều cao, đi bộ tìm biên | `O(n²)` worst (mọi cột bằng nhau) | `O(1)` | Insight "cột thấp nhất" |
| 3. Chia để trị | Cắt tại cột thấp nhất, đệ quy hai bên | `O(n log n)` TB, `O(n²)` worst (mảng sắp xếp) | `O(n)` worst (đệ quy) | Có thể `RecursionError` |
| 4. Hai lượt monotonic stack | `left_smaller`, `right_smaller` rồi `R - L - 1` | `O(n)` | `O(n)` | Rõ ràng, dễ tái dùng |
| 5. Một lượt + sentinel | Tính diện tích ngay lúc pop | `O(n)` | `O(n)` | **Cách chính**, nên nộp |

Lộ trình suy nghĩ: **thử mọi đoạn** → nhận ra chỉ cần **mỗi cột làm chiều cao** → nhận ra chỉ cần **cột thấp hơn gần nhất hai bên** → tính bằng **monotonic stack** → gộp hai lượt thành **một lượt** vì lúc pop đã biết đủ hai biên.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Tính `width = i - top` thay vì `i - stack[-1] - 1`

Sai vì các cột nằm giữa biên trái và `top` có thể **đã bị pop từ trước**, nhưng hình chữ nhật của `top` vẫn lan qua chúng. Ví dụ `[2, 1, 2]`: khi sentinel pop cột 1 (cao 1), `i - top = 3 - 1 = 2`, nhưng đúng phải là cả 3 cột → đáp án sai 2 thay vì 3.

### Lỗi 2: Quên sentinel `0` ở cuối

Các cột chưa gặp cột thấp hơn bên phải sẽ **nằm lại trong stack** và không bao giờ được tính. Mảng tăng dần `[1,2,3,4,5]` không pop lần nào → trả về 0 thay vì 9.

### Lỗi 3: Không có sentinel `-1` (hoặc không kiểm tra stack rỗng) mà vẫn đọc `stack[-1]` sau khi pop

Sau khi pop phần tử cuối cùng, `stack[-1]` trên list rỗng → `IndexError`. Sửa: thêm `-1` ở đáy, **hoặc** viết `width = i if not stack else i - stack[-1] - 1`.

### Lỗi 4: Lan trái quên chặn `left - 1 >= 0` (cách 2)

Python không báo lỗi với `heights[-1]` mà trả về **phần tử cuối** → lan vòng sang cuối mảng. Ví dụ `[3, 1, 3]`: cột 0 "lan" sang `heights[-1] = 3` → tính rộng 2, ra 6 thay vì 3. Lỗi im lặng, rất khó phát hiện!

### Lỗi 5: Nhầm với bài "Container With Most Water" (LeetCode 11)

Ở bài 11, chỉ hai **cột đầu mút** quyết định chiều cao: `min(h[i], h[j]) × (j - i)`. Ở bài 84, **mọi cột nằm giữa** đều chặn trần: chiều cao là `min` của **cả đoạn**. Dùng công thức hai đầu mút cho `[2, 1, 2]` ra `2 × 3 = 6` — sai.

### Lỗi 6: Lưu **chiều cao** vào stack thay vì **index**

Khi pop, ta cần biết vị trí để tính `width`. Lưu chiều cao thì mất thông tin vị trí. Luôn lưu index, chiều cao thì tra `heights[index]`.

In [33]:
def wrong_width_i_minus_top(heights):              # Lỗi 1
    hs = heights + [0]
    stack, best = [-1], 0
    for i, h in enumerate(hs):
        while stack[-1] != -1 and hs[stack[-1]] > h:
            top = stack.pop()
            best = max(best, hs[top] * (i - top))  # SAI: bỏ quên các cột đã pop bên trái top
        stack.append(i)
    return best

def wrong_no_right_sentinel(heights):              # Lỗi 2
    stack, best = [-1], 0
    for i, h in enumerate(heights):                # không có cột 0 ở cuối
        while stack[-1] != -1 and heights[stack[-1]] > h:
            top = stack.pop()
            best = max(best, heights[top] * (i - stack[-1] - 1))
        stack.append(i)
    return best                                    # các cột còn trong stack bị bỏ quên

def wrong_no_left_sentinel(heights):               # Lỗi 3
    hs = heights + [0]
    stack, best = [], 0
    for i, h in enumerate(hs):
        while stack and hs[stack[-1]] > h:
            top = stack.pop()
            best = max(best, hs[top] * (i - stack[-1] - 1))   # stack có thể rỗng!
        stack.append(i)
    return best

def wrong_expand_no_guard(heights):                # Lỗi 4
    best, n = 0, len(heights)
    for k in range(n):
        left = right = k
        # SAI: thiếu left - 1 >= 0 (vế 'left - 1 > -n' chỉ để khỏi IndexError khi demo)
        while heights[left - 1] >= heights[k] and left - 1 > -n:
            left -= 1
        while right + 1 < n and heights[right + 1] >= heights[k]:
            right += 1
        best = max(best, heights[k] * (right - left + 1))
    return best

def wrong_two_ends(heights):                       # Lỗi 5
    n = len(heights)
    return max(min(heights[i], heights[j]) * (j - i + 1) for i in range(n) for j in range(i, n))

print("Lỗi 1 [2,1,2]       :", wrong_width_i_minus_top([2, 1, 2]), "| đúng:", largest_rectangle_stack([2, 1, 2]))
print("Lỗi 2 [1,2,3,4,5]   :", wrong_no_right_sentinel([1, 2, 3, 4, 5]), "| đúng:", largest_rectangle_stack([1, 2, 3, 4, 5]))
try:
    wrong_no_left_sentinel([2, 1])
except IndexError as e:
    print("Lỗi 3 [2,1]         : IndexError:", e)
print("Lỗi 4 [3,1,3]       :", wrong_expand_no_guard([3, 1, 3]), "| đúng:", largest_rectangle_stack([3, 1, 3]))
print("Lỗi 5 [2,1,2]       :", wrong_two_ends([2, 1, 2]), "| đúng:", largest_rectangle_stack([2, 1, 2]))

Lỗi 1 [2,1,2]       : 2 | đúng: 3
Lỗi 2 [1,2,3,4,5]   : 0 | đúng: 9
Lỗi 3 [2,1]         : IndexError: list index out of range
Lỗi 4 [3,1,3]       : 6 | đúng: 3
Lỗi 5 [2,1,2]       : 6 | đúng: 3


## 13. Mẹo nhận dạng pattern và các bài liên quan

### Dấu hiệu nên nghĩ tới monotonic stack

- Đề hỏi tối ưu một đại lượng dạng **`min(đoạn) × độ dài đoạn`** (hoặc `min(đoạn) × tổng đoạn`) trên mọi đoạn con liền nhau → đổi góc nhìn: **"mỗi phần tử làm `min`"**, rồi tìm **đoạn rộng nhất** mà nó còn là `min`.
- Câu hỏi con dạng **"phần tử nhỏ hơn / lớn hơn gần nhất bên trái / bên phải"** → monotonic stack, `O(n)`.
  - tìm **nhỏ hơn** gần nhất → stack **tăng dần** (pop phần tử lớn hơn);
  - tìm **lớn hơn** gần nhất → stack **giảm dần** (pop phần tử nhỏ hơn).
- Brute force có dạng "với mỗi `k`, đi bộ sang trái/phải tới khi gặp phần tử vi phạm" → gần như chắc chắn thay được bằng monotonic stack.
- Thấy cần xử lý "phần còn sót trong stack sau vòng lặp" → nghĩ tới **sentinel** để gộp vào vòng lặp chính.

### Câu thần chú

> **"Cột bị pop là cột vừa biết biên phải; đỉnh mới là biên trái; rộng = i − đỉnh mới − 1."**

### Bài liên quan (luyện tiếp)

| Bài | Liên hệ |
|---|---|
| 85. Maximal Rectangle (Hard) | Mỗi hàng của ma trận 0/1 tạo một histogram → gọi lại bài 84 (Bài tập 5) |
| 1793. Maximum Score of a Good Subarray (Hard) | Giống hệt 84 nhưng đoạn bắt buộc chứa index `k` (Bài tập 6) |
| 42. Trapping Rain Water (Hard) | Histogram + monotonic stack (giảm dần) đổ nước theo lớp |
| 907. Sum of Subarray Minimums | "Mỗi phần tử làm `min`" + `L`, `R` từ monotonic stack, đếm số đoạn `(k - L) × (R - k)` |
| 1856. Maximum Subarray Min-Product | `min(đoạn) × tổng(đoạn)`: monotonic stack + prefix sum |
| 496 / 503. Next Greater Element I / II, 739. Daily Temperatures | Bài nhập môn "phần tử lớn hơn gần nhất" |
| 2104. Sum of Subarray Ranges | Áp dụng hai lần: cho `max` và cho `min` |

## 14. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 15, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay bảng `L`, `R` (không chạy code)

Với `heights = [6, 2, 5, 4, 5, 1, 6]`, hãy lập bảng như mục 7: `L[k]`, `R[k]` (cột **nhỏ hơn hẳn** gần nhất bên trái / phải, quy ước `-1` và `n`), `width = R - L - 1`, `area`. Diện tích lớn nhất là bao nhiêu, và nằm ở cột nào? Vẽ lại bằng `draw_histogram` để kiểm tra.

### Bài 2 — Đoán hành vi của stack một lượt

Chạy tay cách 5 (điều kiện pop là `>`, có sentinel `-1` và `0`) trên `heights = [3, 1, 3, 2, 2]`:

- (a) Tại mỗi `i`, những cột nào bị pop? `L` và `width` của từng lần pop là bao nhiêu?
- (b) Có lần pop nào bị **tính hụt** độ rộng không? Vì sao đáp án vẫn đúng?
- (c) Đáp án cuối cùng?

### Bài 3 — Điền TODO: monotonic stack một lượt

Hoàn thiện hàm theo đúng 4 bước của cách 5. Không nhìn lại code ở mục 9 nhé!

In [34]:
def exercise_largest_rectangle(heights: List[int]) -> int:
    hs = heights + [0]
    stack = [-1]
    best = 0
    for i, h in enumerate(hs):
        # TODO 1: viết vòng while: chừng nào stack[-1] không phải -1 và cột ở đỉnh cao hơn h
        # TODO 2:     pop đỉnh ra biến top
        # TODO 3:     tính width từ i và đỉnh MỚI, cập nhật best
        # TODO 4: push i
        pass
    return best

# Tự làm xong rồi mới mở test:
# assert exercise_largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
# assert exercise_largest_rectangle([2, 4]) == 4
# assert exercise_largest_rectangle([1, 2, 3, 4, 5]) == 9
# assert exercise_largest_rectangle([2, 1, 2]) == 3

### Bài 4 — Tự viết lại cách 4 (hai lượt) trong **một hàm duy nhất**

Không gọi `previous_smaller` / `next_smaller` có sẵn: tự viết hai vòng monotonic stack (một vòng trái → phải, một vòng phải → trái) ngay trong hàm, rồi tính `max(heights[k] × (right[k] - left[k] - 1))`. Sau đó tự trả lời: nếu ở **lượt phải → trái** ta đổi điều kiện pop từ `>=` thành `>`, đáp án có còn đúng không? Thử với `[3, 3]` và `[2, 2, 1, 2, 2]`.

In [35]:
def exercise_two_pass(heights: List[int]) -> int:
    n = len(heights)
    left = [-1] * n
    right = [n] * n
    # TODO 1: vòng i = 0..n-1 với stack, điền left[i] (nhỏ hơn hẳn gần nhất bên trái)
    # TODO 2: vòng i = n-1..0 với stack mới, điền right[i] (nhỏ hơn hẳn gần nhất bên phải)
    # TODO 3: trả về max của heights[k] * (right[k] - left[k] - 1)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_two_pass([2, 1, 5, 6, 2, 3]) == 10
# assert exercise_two_pass([2, 4]) == 4
# assert exercise_two_pass([3, 3]) == 6

### Bài 5 — Biến thể: LeetCode 85 — Maximal Rectangle (Hard)

Cho ma trận nhị phân `rows × cols` gồm các ký tự `"0"` và `"1"`, tìm hình chữ nhật lớn nhất **chỉ chứa `"1"`** và trả về diện tích.

```text
matrix = [["1","0","1","0","0"],
          ["1","0","1","1","1"],
          ["1","1","1","1","1"],
          ["1","0","0","1","0"]]
Output: 6

matrix = [["0"]] -> 0
matrix = [["1"]] -> 1
```

Constraints: `rows == matrix.length`, `cols == matrix[i].length`, `1 <= rows, cols <= 200`, `matrix[i][j]` là `'0'` hoặc `'1'`.

Câu hỏi tư duy: nếu coi **mỗi hàng là đáy**, thì cột `c` "cao" bao nhiêu? Làm sao cập nhật chiều cao từ hàng trên xuống hàng dưới trong `O(1)` mỗi ô?

In [36]:
def exercise_maximal_rectangle(matrix: List[List[str]]) -> int:
    # TODO 1: heights = [0] * số cột
    # TODO 2: với mỗi hàng: cập nhật heights[c] (ô "1" thì +1, ô "0" thì về 0)
    # TODO 3: gọi largest_rectangle_stack(heights), cập nhật best
    pass

# Tự làm xong rồi mới mở test:
# m = [["1","0","1","0","0"], ["1","0","1","1","1"], ["1","1","1","1","1"], ["1","0","0","1","0"]]
# assert exercise_maximal_rectangle(m) == 6
# assert exercise_maximal_rectangle([["0"]]) == 0
# assert exercise_maximal_rectangle([["1"]]) == 1

### Bài 6 — Mini challenge: LeetCode 1793 — Maximum Score of a Good Subarray (Hard)

Cho mảng `nums` và số nguyên `k`. **Score** của đoạn `(i, j)` là `min(nums[i], ..., nums[j]) × (j - i + 1)`. Đoạn là **good** nếu `i <= k <= j`. Trả về score lớn nhất của một đoạn good.

```text
nums = [1,4,3,7,4,5], k = 3 -> 15   (đoạn (1, 5): min = 3, dài 5)
nums = [5,5,4,5,4,1,1,1], k = 0 -> 20   (đoạn (0, 4): min = 4, dài 5)
```

Constraints: `1 <= nums.length <= 10^5`, `1 <= nums[i] <= 2 × 10^4`, `0 <= k < nums.length`.

Nhận ra không: đây **chính là** bài 84 (cột = `nums[i]`), chỉ thêm điều kiện hình chữ nhật phải phủ cột `k`. Yêu cầu: `O(n)`. Câu hỏi tư duy: với mỗi cột làm chiều cao, đoạn lan tối đa `L+1 .. R-1` phải thỏa điều kiện gì để được tính? Vì sao chỉ cần xét **đoạn lan tối đa** mà không bỏ sót đáp án?

In [37]:
def exercise_maximum_score(nums: List[int], k: int) -> int:
    # TODO: tính L, R (nhỏ hơn hẳn gần nhất hai bên) bằng monotonic stack,
    #       chỉ xét cột idx nếu đoạn lan tối đa của nó chứa k
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_maximum_score([1, 4, 3, 7, 4, 5], 3) == 15
# assert exercise_maximum_score([5, 5, 4, 5, 4, 1, 1, 1], 0) == 20

## 15. Gợi ý theo mức

**Bài 1**
- Mức 1: làm theo đúng bảng mục 7; với mỗi `k`, nhìn sang trái tới cột đầu tiên **thấp hơn hẳn**, rồi nhìn sang phải.
- Mức 2: cột 5 cao 1 là cột thấp nhất mảng — nó chặn phải cho mọi cột từ 1 tới 4 (trừ khi có cột thấp hơn gần hơn).
- Mức 3: `L = [-1, -1, 1, 1, 3, -1, 5]`. Tự điền `R` rồi tính `width`.

**Bài 2**
- Mức 1: chiều cao tăng hoặc bằng thì chỉ push; chỉ pop khi gặp cột **thấp hơn hẳn**.
- Mức 2: hai cột cao 2 ở index 3, 4 sẽ cùng nằm trong stack tới sentinel.
- Mức 3: tại `i = 5` (sentinel) có 3 lần pop: cột 4, cột 3, cột 1.

**Bài 3**
- Mức 1: chép lại "thuật toán bằng lời" ở mục 9.
- Mức 2: `while stack[-1] != -1 and hs[stack[-1]] > h:`.
- Mức 3: `top = stack.pop()`, `width = i - stack[-1] - 1`, `best = max(best, hs[top] * width)`, cuối vòng `stack.append(i)`.

**Bài 4**
- Mức 1: hai vòng giống hệt `previous_smaller` và `next_smaller` ở mục 7.
- Mức 2: vòng ngược `for i in range(n - 1, -1, -1):`; stack rỗng thì `right[i]` giữ giá trị mặc định `n`.
- Mức 3: với `>`, cột bên trái trong nhóm bằng nhau nhận `right` là cột bằng nó (hụt), nhưng cột bên phải nhất của nhóm vẫn có `right` đúng và `left` đúng (vì lượt trái dùng `>=`).

**Bài 5**
- Mức 1: đứng ở hàng `r`, chiều cao cột `c` = số ô `"1"` liên tiếp **kéo lên** từ hàng `r`.
- Mức 2: `heights[c] = heights[c] + 1 if row[c] == "1" else 0`.
- Mức 3: mỗi hàng gọi `largest_rectangle_stack(heights)` một lần → tổng `O(rows × cols)`.

**Bài 6**
- Mức 1: hình chữ nhật tối ưu có một cột thấp nhất `idx`; lan nó ra tối đa thì `min` không đổi, độ dài tăng, và **vẫn chứa `k`**.
- Mức 2: cột `idx` có đoạn lan tối đa `L[idx]+1 .. R[idx]-1`; đoạn đó chứa `k` khi `L[idx] < k < R[idx]`.
- Mức 3: dùng lại `previous_smaller`, `next_smaller`; lọc theo `L[idx] < k < R[idx]` rồi lấy max. (Cách khác cũng `O(n)`: two pointers xuất phát từ `k`, mỗi bước mở rộng sang phía có phần tử **lớn hơn**.)

## 16. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1** — `heights = [6, 2, 5, 4, 5, 1, 6]`, `n = 7`:

| k | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `heights[k]` | 6 | 2 | 5 | 4 | 5 | 1 | 6 |
| `L[k]` | -1 | -1 | 1 | 1 | 3 | -1 | 5 |
| `R[k]` | 1 | 5 | 3 | 5 | 5 | 7 | 7 |
| `width` | 1 | 5 | 1 | 3 | 1 | 7 | 1 |
| `area` | 6 | 10 | 5 | **12** | 5 | 7 | 6 |

Đáp án **12**: cột 3 (cao 4) lan được sang cột 2 và cột 4 (đều cao 5 ≥ 4) → phủ cột `2..4`, `4 × 3 = 12`. Để ý cột 1 (cao 2) lan rất rộng (5 cột) nhưng chỉ được 10.

**Bài 2** — `heights = [3, 1, 3, 2, 2]`:

| i | h | pop | L | width | area | best |
|---:|---:|---|---:|---:|---:|---:|
| 1 | 1 | cột 0 (h=3) | -1 | 1 | 3 | 3 |
| 3 | 2 | cột 2 (h=3) | 1 | 1 | 3 | 3 |
| 5 | 0 (sentinel) | cột 4 (h=2) | 3 | 1 | 2 | 3 |
| 5 | 0 (sentinel) | cột 3 (h=2) | 1 | 3 | **6** | 6 |
| 5 | 0 (sentinel) | cột 1 (h=1) | -1 | 5 | 5 | 6 |

- (b) Lần pop cột 4 bị **tính hụt** (rộng 1 thay vì 2), vì cột 3 cùng cao 2 nằm ngay dưới nó trong stack nên bị coi là "biên trái". Nhưng ngay sau đó cột 3 bị pop với `L = 1` và được rộng đủ 3 (cột 2..4) → `2 × 3 = 6`.
- (c) Đáp án **6**.

In [38]:
# Kiểm chứng Bài 1 và Bài 2
h_ex1 = [6, 2, 5, 4, 5, 1, 6]
assert previous_smaller(h_ex1) == [-1, -1, 1, 1, 3, -1, 5]
assert next_smaller(h_ex1) == [1, 5, 3, 5, 5, 7, 7]
assert largest_rectangle_stack(h_ex1) == 12
draw_histogram(h_ex1, rect=(2, 4, 4))
print()
assert trace_stack([3, 1, 3, 2, 2]) == 6

 6 |##          ##
 5 |##  ##  ##  ##
 4 |##  @@@@@@  ##
 3 |##  @@@@@@  ##
 2 |####@@@@@@  ##
 1 |####@@@@@@####
   +--------------
    0 1 2 3 4 5 6 

i=0 h=3: push 0 -> stack=[-1, 0]
i=1 h=1: pop 0 (h=3), L=-1, width=1-(-1)-1=1, area= 3, best= 3, stack=[-1]
i=1 h=1: push 1 -> stack=[-1, 1]
i=2 h=3: push 2 -> stack=[-1, 1, 2]
i=3 h=2: pop 2 (h=3), L= 1, width=3-1-1=1, area= 3, best= 3, stack=[-1, 1]
i=3 h=2: push 3 -> stack=[-1, 1, 3]
i=4 h=2: push 4 -> stack=[-1, 1, 3, 4]
i=5 h=0 (sentinel): pop 4 (h=2), L= 3, width=5-3-1=1, area= 2, best= 3, stack=[-1, 1, 3]
i=5 h=0 (sentinel): pop 3 (h=2), L= 1, width=5-1-1=3, area= 6, best= 6, stack=[-1, 1]
i=5 h=0 (sentinel): pop 1 (h=1), L=-1, width=5-(-1)-1=5, area= 5, best= 6, stack=[-1]
i=5 h=0 (sentinel): push 5 -> stack=[-1, 5]


In [39]:
# Đáp án Bài 3
def answer_largest_rectangle(heights: List[int]) -> int:
    hs = heights + [0]
    stack = [-1]
    best = 0
    for i, h in enumerate(hs):
        while stack[-1] != -1 and hs[stack[-1]] > h:
            top = stack.pop()
            width = i - stack[-1] - 1
            best = max(best, hs[top] * width)
        stack.append(i)
    return best

assert answer_largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
assert answer_largest_rectangle([2, 4]) == 4
assert answer_largest_rectangle([1, 2, 3, 4, 5]) == 9
assert answer_largest_rectangle([2, 1, 2]) == 3


# Đáp án Bài 4
def answer_two_pass(heights: List[int], right_pop_strict=False) -> int:
    n = len(heights)
    left = [-1] * n
    right = [n] * n
    stack = []
    for i in range(n):
        while stack and heights[stack[-1]] >= heights[i]:
            stack.pop()
        left[i] = stack[-1] if stack else -1
        stack.append(i)
    stack = []
    for i in range(n - 1, -1, -1):
        if right_pop_strict:                      # biến thể: pop khi '>' (để trả lời câu hỏi phụ)
            while stack and heights[stack[-1]] > heights[i]:
                stack.pop()
        else:
            while stack and heights[stack[-1]] >= heights[i]:
                stack.pop()
        right[i] = stack[-1] if stack else n
        stack.append(i)
    return max(heights[k] * (right[k] - left[k] - 1) for k in range(n))

assert answer_two_pass([2, 1, 5, 6, 2, 3]) == 10
assert answer_two_pass([2, 4]) == 4
assert answer_two_pass([3, 3]) == 6
# Câu hỏi phụ: đổi lượt phải sang '>' vẫn đúng
for h in ([3, 3], [2, 2, 1, 2, 2]):
    print(h, "-> '>=':", answer_two_pass(h), "| '>':", answer_two_pass(h, right_pop_strict=True))
random.seed(5)
for _ in range(500):
    h = [random.randint(0, 4) for _ in range(random.randint(1, 8))]
    assert answer_two_pass(h, right_pop_strict=True) == largest_rectangle_naive(h)
print("Bài 3, Bài 4 PASS (kể cả biến thể '>' trên 500 mảng ngẫu nhiên)")

[3, 3] -> '>=': 6 | '>': 6
[2, 2, 1, 2, 2] -> '>=': 5 | '>': 5
Bài 3, Bài 4 PASS (kể cả biến thể '>' trên 500 mảng ngẫu nhiên)


**Bài 4 — câu hỏi phụ:** vẫn đúng. Với `>` ở lượt phải, trong một nhóm cột bằng nhau, cột bên trái nhận `right` là cột bằng nó (hụt), nhưng **cột bên phải nhất** của nhóm vẫn có `right` là cột nhỏ hơn hẳn thật sự, còn `left` của nó (lượt trái dùng `>=`) là cột nhỏ hơn hẳn thật sự → nó nhận đủ độ rộng. Ví dụ `[3, 3]`: cột 0 được rộng 1, cột 1 được rộng 2 → max vẫn 6.

**Bài 5 — Maximal Rectangle:** đứng ở hàng `r`, chiều cao cột `c` là số ô `"1"` liên tiếp tính từ hàng `r` **ngược lên trên**. Với ví dụ:

| hàng | `heights` sau khi cập nhật | bài 84 cho |
|---:|---|---:|
| 0 | `[1, 0, 1, 0, 0]` | 1 |
| 1 | `[2, 0, 2, 1, 1]` | 3 |
| 2 | `[3, 1, 3, 2, 2]` | **6** (chính là Bài 2!) |
| 3 | `[4, 0, 0, 3, 0]` | 4 |

Time `O(rows × cols)`, Space `O(cols)`.

**Bài 6 — Maximum Score of a Good Subarray:** đoạn good tối ưu có một phần tử nhỏ nhất `nums[idx]`. Lan đoạn đó ra tối đa (`L[idx]+1 .. R[idx]-1`) thì `min` vẫn là `nums[idx]`, độ dài chỉ tăng, và đoạn mới **chứa đoạn cũ** nên vẫn chứa `k`. Vậy chỉ cần xét các đoạn lan tối đa thỏa `L[idx] < k < R[idx]`. Time `O(n)`, Space `O(n)`.

In [40]:
# Đáp án Bài 5
def answer_maximal_rectangle(matrix: List[List[str]]) -> int:
    if not matrix:
        return 0
    cols = len(matrix[0])
    heights = [0] * cols
    best = 0
    for row in matrix:
        for c in range(cols):
            heights[c] = heights[c] + 1 if row[c] == "1" else 0
        best = max(best, largest_rectangle_stack(heights))
    return best

m = [["1", "0", "1", "0", "0"],
     ["1", "0", "1", "1", "1"],
     ["1", "1", "1", "1", "1"],
     ["1", "0", "0", "1", "0"]]
assert answer_maximal_rectangle(m) == 6
assert answer_maximal_rectangle([["0"]]) == 0
assert answer_maximal_rectangle([["1"]]) == 1


# Đáp án Bài 6
def answer_maximum_score(nums: List[int], k: int) -> int:
    L = previous_smaller(nums)
    R = next_smaller(nums)
    best = 0
    for idx in range(len(nums)):
        if L[idx] < k < R[idx]:                      # đoạn lan tối đa có chứa k
            best = max(best, nums[idx] * (R[idx] - L[idx] - 1))
    return best

def maximum_score_bruteforce(nums, k):
    best = 0
    for i in range(k + 1):
        for j in range(k, len(nums)):
            best = max(best, min(nums[i:j + 1]) * (j - i + 1))
    return best

assert answer_maximum_score([1, 4, 3, 7, 4, 5], 3) == 15
assert answer_maximum_score([5, 5, 4, 5, 4, 1, 1, 1], 0) == 20
random.seed(1793)
for _ in range(500):
    nums = [random.randint(1, 6) for _ in range(random.randint(1, 9))]
    k = random.randrange(len(nums))
    assert answer_maximum_score(nums, k) == maximum_score_bruteforce(nums, k), (nums, k)
print("Bài 5, Bài 6 PASS (Bài 6 đối chiếu brute force trên 500 mảng ngẫu nhiên)")

Bài 5, Bài 6 PASS (Bài 6 đối chiếu brute force trên 500 mảng ngẫu nhiên)


## 17. Template ngắn nên nhớ

```python
class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        n = len(heights)
        stack = [-1]                                  # sentinel trái: bức tường ảo
        best = 0
        for i in range(n + 1):
            h = heights[i] if i < n else 0            # sentinel phải: cột cao 0 dọn sạch stack
            while stack[-1] != -1 and heights[stack[-1]] > h:
                top = stack.pop()                     # top vừa biết biên phải = i
                width = i - stack[-1] - 1             # biên trái = đỉnh mới
                best = max(best, heights[top] * width)
            stack.append(i)
        return best
```

Câu thần chú: **"Mỗi cột thử làm chiều cao → cần cột thấp hơn gần nhất hai bên → monotonic stack tăng dần; pop là lúc tính, rộng = i − đỉnh mới − 1."**

## Checklist trước khi submit

- [ ] Stack lưu **index**, không lưu chiều cao.
- [ ] Có sentinel trái `-1` (hoặc xử lý stack rỗng: `width = i`), không bao giờ pop `-1`.
- [ ] Có sentinel phải chiều cao `0` tại `i = n` để dọn hết các cột còn chờ — thử ngay `[1, 2, 3, 4, 5]` (phải ra 9).
- [ ] `width = i - stack[-1] - 1` tính **sau khi** pop, dùng đỉnh **mới**, không phải `i - top`.
- [ ] Hiểu điều kiện pop `>` hay `>=` và vì sao cột bằng nhau vẫn cho đáp án đúng — thử `[3, 3]` (phải ra 6).
- [ ] Không sửa list input (hoặc nếu `append(0)` thì nhớ `pop()` trả lại).
- [ ] Test các ca: `[0]`, một cột, toàn bằng nhau, tăng dần, giảm dần, có cột `0` ở giữa, `[2, 1, 2]`.
- [ ] Độ phức tạp: `O(n)` time (mỗi index push 1 lần, pop tối đa 1 lần), `O(n)` space.